In [0]:
from common_utils.logging import get_logger
logger = get_logger("ds2b_s3")

import pyspark.sql.functions as F 

logger.info("Defining raw path")
raw_path="/Volumes/retaildataplatform/bronze/raw_data/sales/load_date=2026-09-14/"
logger.info("raw path given is %s",raw_path)

logger.info("Reading data")
df=spark.read.format("delta").option("header","true").option("inferSchema","true").load(raw_path)

logger.info("sample data %s",df.show(5))

logger.info("Auditing columns")

df=df.withColumn("last_upadte_ts",F.current_timestamp())\
    .withColumn("file_path",F.col("_metadata.file_path"))
target_path=f"retaildataplatform.bronze.s3server_customers"

logger.info("Target path is %s",target_path)

logger.info("Writing data")
df.write.mode("overwrite").format("delta").saveAsTable(target_path)
logger.info("Data written successfully at%s",target_path)